In [0]:
df = spark.read.csv("/Volumes/inventoryoptimizationstocks/default/inventory/stock_dataset_dirty_212_rows.csv", header=True, inferSchema=True)
df.show()


+----------+------------+-----------+--------------+-------------+----------------+-------------+----------------------+---------------+-------------+----+----+------------+
|Product_ID|     Produit|  Categorie|Quantite_Stock|Prix_Unitaire|Demande_Annuelle|Cout_Commande|Cout_Stockage_Unitaire|Lead_Time_Jours|  Fournisseur|_c10|_c11|        _c12|
+----------+------------+-----------+--------------+-------------+----------------+-------------+----------------------+---------------+-------------+----+----+------------+
|      P031|      Filtre| Production|          1635|       1821.8|            9204|       158.08|                 28.12|              3|Maghreb Parts|NULL|NULL|      Filtre|
|      P174|   Roulement|Maintenance|          1150|      1489.89|            9712|       376.24|                 21.56|             37|Maghreb Parts|NULL|NULL|   Roulement|
|      P141|       Pompe| Production|          1219|      2261.24|            6803|        23.24|                 49.55|          

In [0]:
df.printSchema()
df.count()

root
 |-- Product_ID: string (nullable = true)
 |-- Produit: string (nullable = true)
 |-- Categorie: string (nullable = true)
 |-- Quantite_Stock: integer (nullable = true)
 |-- Prix_Unitaire: double (nullable = true)
 |-- Demande_Annuelle: integer (nullable = true)
 |-- Cout_Commande: double (nullable = true)
 |-- Cout_Stockage_Unitaire: double (nullable = true)
 |-- Lead_Time_Jours: integer (nullable = true)
 |-- Fournisseur: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)



212

In [0]:
from pyspark.sql.functions import col, when

# Supprimer doublons
df = df.dropDuplicates(["Product_ID"])

# Corriger stock négatif
df = df.withColumn(
    "Quantite_Stock",
    when(col("Quantite_Stock") < 0, 0).otherwise(col("Quantite_Stock"))
)

# Remplacer prix = 0
df = df.withColumn(
    "Prix_Unitaire",
    when(col("Prix_Unitaire") == 0, None).otherwise(col("Prix_Unitaire"))
)

In [0]:
from pyspark.sql import functions as F

# 1) supprimer doublons complets
df_clean = df.dropDuplicates()

# 2) supprimer doublons sur Product_ID
df_clean = df_clean.dropDuplicates(["Product_ID"])

# 3) corriger stock négatif
df_clean = df_clean.withColumn(
    "Quantite_Stock",
    F.when(F.col("Quantite_Stock") < 0, 0).otherwise(F.col("Quantite_Stock"))
)

# 4) remplacer prix = 0 par null
df_clean = df_clean.withColumn(
    "Prix_Unitaire",
    F.when(F.col("Prix_Unitaire") == 0, None).otherwise(F.col("Prix_Unitaire"))
)

# 5) calculer moyennes numériques
avg_prix = df_clean.select(F.avg("Prix_Unitaire")).collect()[0][0]
avg_demande = df_clean.select(F.avg("Demande_Annuelle")).collect()[0][0]
avg_stockage = df_clean.select(F.avg("Cout_Stockage_Unitaire")).collect()[0][0]
avg_lead = df_clean.select(F.avg("Lead_Time_Jours")).collect()[0][0]

# 6) mode colonnes catégorielles
mode_categorie = (
    df_clean.groupBy("Categorie").count()
    .orderBy(F.desc("count"))
    .first()[0]
)

mode_fournisseur = (
    df_clean.groupBy("Fournisseur").count()
    .orderBy(F.desc("count"))
    .first()[0]
)

# 7) remplir les valeurs manquantes
df_clean = df_clean.fillna({
    "Categorie": mode_categorie,
    "Fournisseur": mode_fournisseur,
    "Prix_Unitaire": avg_prix,
    "Demande_Annuelle": avg_demande,
    "Cout_Stockage_Unitaire": avg_stockage,
    "Lead_Time_Jours": avg_lead
})

df_clean.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Produit: string (nullable = true)
 |-- Categorie: string (nullable = false)
 |-- Quantite_Stock: integer (nullable = true)
 |-- Prix_Unitaire: double (nullable = false)
 |-- Demande_Annuelle: integer (nullable = true)
 |-- Cout_Commande: double (nullable = true)
 |-- Cout_Stockage_Unitaire: double (nullable = false)
 |-- Lead_Time_Jours: integer (nullable = true)
 |-- Fournisseur: string (nullable = false)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_abc = (
    df_clean
    .withColumn("Valeur_Stock", F.col("Quantite_Stock") * F.col("Prix_Unitaire"))
    .withColumn("Consommation_Annuelle_Valeur", F.col("Demande_Annuelle") * F.col("Prix_Unitaire"))
)

window_spec = Window.orderBy(
    F.col("Consommation_Annuelle_Valeur").desc(),
    F.col("Product_ID").asc()
)
window_cum = window_spec.rowsBetween(Window.unboundedPreceding, Window.currentRow)

total_valeur = df_abc.agg(
    F.sum("Consommation_Annuelle_Valeur").alias("total")
).collect()[0]["total"]

df_abc = (
    df_abc
    .withColumn("Pourcentage", F.col("Consommation_Annuelle_Valeur") / F.lit(total_valeur))
    .withColumn("Pourcentage_Cumule", F.sum("Pourcentage").over(window_cum))
    .withColumn(
        "Classe_ABC",
        F.when(F.col("Pourcentage_Cumule") <= 0.80, "A")
         .when(F.col("Pourcentage_Cumule") <= 0.95, "B")
         .otherwise("C")
    )
)

df_abc.groupBy("Classe_ABC").count().show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----+
|Classe_ABC|count|
+----------+-----+
|         A|   89|
|         B|   51|
|         C|   60|
+----------+-----+



In [0]:
df_clean.printSchema()
df_abc.groupBy("Classe_ABC").count().show()

root
 |-- Product_ID: string (nullable = true)
 |-- Produit: string (nullable = true)
 |-- Categorie: string (nullable = false)
 |-- Quantite_Stock: integer (nullable = true)
 |-- Prix_Unitaire: double (nullable = false)
 |-- Demande_Annuelle: integer (nullable = true)
 |-- Cout_Commande: double (nullable = true)
 |-- Cout_Stockage_Unitaire: double (nullable = false)
 |-- Lead_Time_Jours: integer (nullable = true)
 |-- Fournisseur: string (nullable = false)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----+
|Classe_ABC|count|
+----------+-----+
|         A|   89|
|         B|   51|
|         C|   60|
+----------+-----+

